# Chapter 4 — Round Persistence and ProofFrame Diff

Two routemap-window capabilities that turn capability calls into a durable audit trail and a comparable view across runs:

1. **Round Persistence** (Batch 6) — capture a sequence of capability results inside an audit package via the external recorder API (`start_round`, `record_round_event`, `finalize_round`). The recorder writes `audit/round_events.jsonl` atomically (tempfile + `os.replace`) at finalize time.
2. **ProofFrame Diff** (Batch 7) — given two finalized rounds, `AuditQuery.diff_proof_frames(...)` derives a per-frame delta over their `proof_frame_result` events. Frame identity is `(support_digest, binding_items)`; per-atom identity is `atom_key` within the same `support_digest`.

**Series navigation**

- Previous: `03_proofframe_rule_overlays.ipynb`.
- This is the final chapter. The integrated walkthrough is `round_story_full_demo.py` (run as a script for end-to-end smoke verification).

## Setup

In [ ]:
import sys
from pathlib import Path

_repo_root = Path.cwd()
if not (_repo_root / 'examples').exists() and (_repo_root.parent / 'examples').exists():
    _repo_root = _repo_root.parent
for sub in ('src', 'examples'):
    candidate = _repo_root / sub
    if candidate.exists() and str(candidate) not in sys.path:
        sys.path.insert(0, str(candidate))

import round_story_full_demo as demo  # noqa: E402

## 1. Round Recorder API (Batch 6)

The external recorder lives at `kernel.audit.round_events`:

```python
recorder = start_round(round_id, event_ts=...)
record_round_event(recorder, kind=..., payload=..., event_ts=...)
finalize_round(recorder, package_dir, event_ts=...)
```

Capability runtimes do **not** import `kernel.audit`. The caller invokes a capability, then explicitly records the result outside the runtime — preserving the application↛audit one-direction invariant.

First-slice persisted event kinds:

- lifecycle: `round_started`, `round_finalized`
- capability: `check_result`, `diagnose_result`, `fact_overlay_result`, `why_not_result`, `proof_frame_result`

Frontier projection and rule-action result events are deferred per blueprint §5.5.4.

## 2. ProofFrame Diff API (Batch 7)

`AuditQuery.diff_proof_frames(round_a, round_b, *, include_partial=False, include_unchanged=False)` is a query-derived surface — it does **not** add a new durable file. It consumes only `proof_frame_result` rows.

Key contract points:

- Frame identity: `(payload.request.support_digest, payload.result.binding_items)`.
- Atom identity: `atom_key`, scoped to the same `support_digest`.
- `affected_action_indices` are **not** compared across rounds (per-event semantics only).
- Partial rounds (no `round_finalized` marker) are rejected by default; opt in with `include_partial=True`.
- `future:proof_frame_result` rows (schema_version >= 2.0) are skipped with a `DIFF_FUTURE_KIND_SKIPPED` warning.
- Frames with empty `atom_verdicts` are marked `rule_refs_unsupported` and produce no per-atom delta.

## 3. End-to-end: two rounds + diff

`run_round_persistence_diff_demo(...)` exercises both APIs together inside a temporary audit package:

1. Build the seed fixture and run the prerequisite Check / Diagnose / Fact Overlay / Why-not / ProofFrame phases (silenced).
2. Start a `round-baseline` round, record 5 events (one per capability above using the shipped `project_*_event_payload` helpers), then `finalize_round` (atomic write).
3. Start a `round-overlay` round, record one `proof_frame_result` event (the overlay-driven ProofFrame), finalize.
4. Load the audit package, list rounds, call `diff_proof_frames('round-baseline', 'round-overlay')`.
5. Assert one `FrameDelta` with `frame_status_change.before == 'still_valid'` and `frame_status_change.after == 'invalidated'`.

All of this is contained in `_phase_round_persistence_and_diff` so the notebook does not fork behavior.

In [ ]:
summary = demo.run_round_persistence_diff_demo(verbose=True)
expected = {'round_diff': 'frame_status_changed'}

print(f'\nChapter summary : {summary}')
print(f'Expected        : {expected}')
assert summary == expected, summary
print('\n✓ Chapter 4 aggregate matches expected smoke contract.')

## 4. Inspecting the round events in isolation

The bundled helper above tore down the temp audit package on exit. To inspect the persisted events directly, the cell below replays the same flow but keeps the package in `/tmp` so we can read the JSONL rows.

In [ ]:
import tempfile
from kernel.audit import AuditQuery, load_audit_package

fixture = demo._build_fixture()
check_request, check_result, support = demo._phase_check(fixture, verbose=False)
_dr, _diagnose_result = demo._phase_diagnose(fixture, verbose=False)
_fr, _fact_overlay_result, fact_overlay = demo._phase_fact_overlay(fixture, verbose=False)
_baseline_pf_request, _baseline_pf_result, overlay_pf_request, overlay_pf_result = (
    demo._phase_proofframe(fixture, support, fact_overlay, verbose=False)
)

with tempfile.TemporaryDirectory() as tmpdir:
    package_dir = demo._minimal_audit_package(Path(tmpdir) / 'audit_pkg')
    package = load_audit_package(package_dir)
    print(f'Empty package round_events count: {len(package.round_events)}')
    # Mini round demonstrating the recorder API directly.
    from kernel.audit.round_events import (
        finalize_round,
        project_proof_frame_event_payload,
        record_round_event,
        start_round,
    )
    recorder = start_round('round-mini', event_ts=1000)
    record_round_event(
        recorder,
        kind='proof_frame_result',
        payload=project_proof_frame_event_payload(overlay_pf_request, overlay_pf_result),
        event_ts=1001,
    )
    finalize_round(recorder, package_dir, event_ts=1002)
    package = load_audit_package(package_dir)
    query = AuditQuery(package)
    print(f'Rounds after finalize: {query.list_rounds()}')
    summary = query.get_round_summary('round-mini')
    print(f'round-mini summary: is_finalized={summary.is_finalized} '
          f'event_count={summary.event_count} kinds={summary.kind_counts}')

## Where to next

- **Integrated walkthrough** — `round_story_full_demo.py` (run as `python examples/round_story_full_demo.py`) executes all four chapters back-to-back and asserts the full `EXPECTED_PHASE_SUMMARY` contract.
- **Module references** — `src/kernel/audit/docs/01_overview.md` (recorder + query layer) and `src/kernel/audit/docs/03_audit_package_contract.md` (round_events.jsonl row shape, lenient reader rules).
- **Boundary** — Round persistence and ProofFrame diff are advanced-importable (`kernel.audit.round_events` and `kernel.audit.proof_frame_diff`); v0.1 ships no SDK shells or service routes for them per the Batch 8 public-surface decision.